# GameForge3D — Phase 3: 3D Shape Generator (Shap-E)

**FYP 2026-2027 | NUML Dept. of Computer Science**  
**Supervisor:** Ms. Tooba Sagheer

This notebook uses **OpenAI Shap-E** (open-source, pre-trained) to generate  
game-asset 3D meshes from text prompts routed by our Phase 2 classifier.

**Steps:**
1. Mount Google Drive
2. Install Shap-E + dependencies
3. Load Shap-E text-to-3D model
4. Generate 3D shapes for each asset category
5. Decode latents → mesh (.obj)
6. Save outputs to Drive
7. Integrate with Phase 2 Router (end-to-end pipeline test)

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/GameForge3D/checkpoints'
DATA_DIR       = '/content/drive/MyDrive/GameForge3D/data'
OUTPUT_DIR     = '/content/drive/MyDrive/GameForge3D/outputs/shapes'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive mounted.')
print(f'Outputs will be saved to: {OUTPUT_DIR}')

## Step 2 — Install Shap-E & Dependencies

In [ ]:
!pip install -q git+https://github.com/openai/shap-e.git
!pip install -q trimesh matplotlib torch transformers
print('All packages installed.')

## Step 3 — Load Shap-E Text-to-3D Model

> This downloads pre-trained weights (~1.5 GB). Takes 2-3 minutes.

In [ ]:
import torch
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Loading Shap-E transmitter model...')
xm = load_model('transmitter', device=device)

print('Loading Shap-E text300M model...')
model = load_model('text300M', device=device)

print('Loading diffusion config...')
diffusion = diffusion_from_config(load_config('diffusion'))

print('All models loaded successfully!')

## Step 4 — Generate 3D Shapes for Each Category

We generate one shape per category using representative prompts.

In [ ]:
from shap_e.util.notebooks import create_pan_cameras, decode_latent_images
import matplotlib.pyplot as plt

# Representative prompt for each category
PROMPTS = {
    'Weapon'  : 'a sharp medieval sword with a golden handle',
    'Vehicle' : 'a futuristic racing car with large wheels',
    'Creature': 'a dragon with wings and sharp claws',
    'Prop'    : 'a wooden barrel'
}

LATENTS = {}   # store latents for each category

BATCH_SIZE    = 1
GUIDANCE_SCALE = 15.0

for category, prompt in PROMPTS.items():
    print(f'\nGenerating [{category}]: "{prompt}"')
    latents = sample_latents(
        batch_size      = BATCH_SIZE,
        model           = model,
        diffusion       = diffusion,
        guidance_scale  = GUIDANCE_SCALE,
        model_kwargs    = dict(texts=[prompt] * BATCH_SIZE),
        progress        = True,
        clip_denoised   = True,
        use_fp16        = True,
        use_karras      = True,
        karras_steps    = 64,
        sigma_min       = 1e-3,
        sigma_max       = 160,
        s_churn         = 0
    )
    LATENTS[category] = latents
    print(f'  Done. Latent shape: {latents.shape}')

print('\nAll shapes generated!')

## Step 5 — Preview Renders (360° Views)

In [ ]:
cameras = create_pan_cameras(64, device)

fig, axes = plt.subplots(len(PROMPTS), 4, figsize=(16, len(PROMPTS) * 4))

for row_idx, (category, latents) in enumerate(LATENTS.items()):
    images = decode_latent_images(xm, latents[0], cameras, rendering_mode='nerf')
    for col_idx, img in enumerate(images[:4]):
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx][col_idx].set_title(f'{category}\n{PROMPTS[category][:30]}...',
                                              fontsize=9, loc='left')

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/shape_previews.png', dpi=150)
plt.show()
print('Preview saved.')

## Step 6 — Export Meshes as .obj Files

In [ ]:
from shap_e.util.notebooks import decode_latent_mesh
import trimesh

SAVED_MESHES = {}

for category, latents in LATENTS.items():
    print(f'Exporting [{category}]...')

    # Decode latent to TriMesh
    t = decode_latent_mesh(xm, latents[0]).tri_mesh()

    # Convert to trimesh object
    mesh = trimesh.Trimesh(
        vertices = t.verts,
        faces    = t.faces
    )

    # Center and normalize
    mesh.apply_translation(-mesh.centroid)
    scale = 1.0 / mesh.scale
    mesh.apply_scale(scale)

    # Save as .obj
    slug      = PROMPTS[category].replace(' ', '_')[:30]
    obj_path  = f'{OUTPUT_DIR}/{category.lower()}_{slug}.obj'
    mesh.export(obj_path)

    SAVED_MESHES[category] = obj_path
    verts = len(mesh.vertices)
    faces = len(mesh.faces)
    size  = os.path.getsize(obj_path) // 1024
    print(f'  Saved: {obj_path}')
    print(f'  Vertices: {verts:,} | Faces: {faces:,} | Size: {size} KB')

print('\nAll meshes exported!')

## Step 7 — End-to-End Pipeline Test

Router (Phase 2) → Shape Generator (Phase 3)  
User types a prompt → Router classifies → Shape generated.

In [ ]:
from transformers import pipeline as hf_pipeline

ROUTER_PATH = f'{CHECKPOINT_DIR}/router/distilbert_router_v1'
router = hf_pipeline(
    'text-classification',
    model     = ROUTER_PATH,
    tokenizer = ROUTER_PATH,
    device    = 0 if torch.cuda.is_available() else -1
)

def gameforge_generate(prompt, guidance_scale=15.0, steps=64):
    """Full pipeline: text -> category -> 3D mesh"""

    # Step 1: Route
    route   = router(prompt)[0]
    cat     = route['label']
    conf    = route['score']
    print(f'Router  -> {cat} ({conf:.1%})')

    # Step 2: Generate shape
    print(f'Generating 3D shape for: "{prompt}"')
    latents = sample_latents(
        batch_size     = 1,
        model          = model,
        diffusion      = diffusion,
        guidance_scale = guidance_scale,
        model_kwargs   = dict(texts=[prompt]),
        progress       = True,
        clip_denoised  = True,
        use_fp16       = True,
        use_karras     = True,
        karras_steps   = steps,
        sigma_min      = 1e-3,
        sigma_max      = 160,
        s_churn        = 0
    )

    # Step 3: Export mesh
    t    = decode_latent_mesh(xm, latents[0]).tri_mesh()
    mesh = trimesh.Trimesh(vertices=t.verts, faces=t.faces)
    mesh.apply_translation(-mesh.centroid)
    mesh.apply_scale(1.0 / mesh.scale)

    slug     = prompt.replace(' ', '_')[:25]
    out_path = f'{OUTPUT_DIR}/pipeline_{cat.lower()}_{slug}.obj'
    mesh.export(out_path)
    print(f'Mesh saved -> {out_path}')
    print(f'Vertices: {len(mesh.vertices):,} | Faces: {len(mesh.faces):,}')

    # Step 4: Preview
    cameras = create_pan_cameras(64, device)
    images  = decode_latent_images(xm, latents[0], cameras, rendering_mode='nerf')
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, img in zip(axes, images[:4]):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle(f'{cat} | "{prompt}"', fontsize=11)
    plt.tight_layout()
    plt.show()

    return out_path

# --- Test the pipeline ---
TEST_PROMPT = 'a battle axe with glowing runes'
gameforge_generate(TEST_PROMPT)

## Phase 3 Complete! ✅

| Output | Location |
|--------|----------|
| Shape Previews (4 categories) | `data/shape_previews.png` |
| Weapon .obj mesh | `outputs/shapes/weapon_*.obj` |
| Vehicle .obj mesh | `outputs/shapes/vehicle_*.obj` |
| Creature .obj mesh | `outputs/shapes/creature_*.obj` |
| Prop .obj mesh | `outputs/shapes/prop_*.obj` |
| Pipeline test mesh | `outputs/shapes/pipeline_*.obj` |

**Pipeline so far:**
```
Text Prompt
    ↓  Phase 2: DistilBERT Router
Category (Weapon / Vehicle / Creature / Prop)
    ↓  Phase 3: Shap-E Diffusion
3D Mesh (.obj)
```

**Next → Phase 4: Texture Generator (U-Net)**